# Used words for this model prediction

In [ ]:
# model	  ------   Avtomobilning aniq nomi yoki modeli.

# model_index	----------   Modelning indeks yoki kod raqami (ma’lumotlar bazasida identifikatsiya uchun).

# displacement	--------    Dvigatel hajmi (odatda litr yoki kub santimetrda, masalan 2.0L).

# cylinders	---------    Dvigateldagi silindrlar soni (4, 6, 8 va hokazo).

# gears	---------    Uzatmalar qutisidagi tezliklar soni (masalan, 5 pog‘onali).

# transmission	-------   Uzatmalar qutisi turi (manual — qo‘lda, automatic — avtomat).

# mpg	--------    “Miles per gallon” — yonilg‘i tejamkorligi (1 gallon yonilg‘ida necha mil yuradi).

# aspiration	--------   Havo kirish turi: tabiiy (NA — naturally aspirated) yoki majburiy (turbo/supercharger).

# lockup_torque_converter	drive	--------  Avtomatik transmissiyada “lock-up” funksiyali moment o‘zgartirgich mavjudligi.

# max_ethanol   ---------    Yonilg‘idagi maksimal etanol foizi (masalan, E10 — 10% etanol).

# recommended_fuel	--------   Ishlab chiqaruvchi tavsiya qilgan yonilg‘i turi (Regular, Premium va hokazo).

# intake_valves_per_cyl	--------    Har bir silindrga to‘g‘ri keladigan kirish (havo/yonilg‘i) klapanlari soni.

# exhaust_valves_per_cyl	--------   Har bir silindrga to‘g‘ri keladigan chiqish (gaz) klapanlari soni.

# fuel_injection   ---------    Yonilg‘i purkash tizimi turi (port injection, direct injection va boshqalar).

In [10]:
import pandas as pd
import numpy as np

from joblib import dump
import os

import matplotlib.pyplot as plt
import seaborn as sns
import klib

from sklearn.model_selection import train_test_split, KFold,cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
labeller = LabelEncoder()

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

RF_Clas = RandomForestClassifier()
DT_Clas = DecisionTreeClassifier()
svc = SVC()

from sklearn.metrics import classification_report,accuracy_score

In [48]:
df = pd.read_csv(r"C:\Users\bunyo\Downloads\Telegram Desktop\cars2018.csv")

In [11]:
df.sample(3)

,model,model_index,displacement,cylinders,gears,transmission,mpg,aspiration,lockup_torque_converter,drive,max_ethanol,recommended_fuel,intake_valves_per_cyl,exhaust_valves_per_cyl,fuel_injection
510,NISSAN SENTRA,128,1.6,4,7,CVT,29,Turbocharged/Supercharged,Y,"2-Wheel Drive, Front",15,Premium Unleaded Recommended,2,2,Direct ignition
98,Porsche 911 Carrera 4S Cabriolet,116,3.0,6,7,Manual,24,Turbocharged/Supercharged,N,4-Wheel Drive,10,Premium Unleaded Required,2,2,Direct ignition
1019,Subaru FORESTER,16,2.5,4,6,Manual,24,Naturally Aspirated,N,All Wheel Drive,10,Regular Unleaded Recommended,2,2,Multipoint/sequential ignition


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1144 entries, 0 to 1143
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   model                    1144 non-null   object 
 1   model_index              1144 non-null   int64  
 2   displacement             1144 non-null   float64
 3   cylinders                1144 non-null   int64  
 4   gears                    1144 non-null   int64  
 5   transmission             1144 non-null   object 
 6   mpg                      1144 non-null   int64  
 7   aspiration               1144 non-null   object 
 8   lockup_torque_converter  1144 non-null   object 
 9   drive                    1144 non-null   object 
 10  max_ethanol              1144 non-null   int64  
 11  recommended_fuel         1144 non-null   object 
 12  intake_valves_per_cyl    1144 non-null   int64  
 13  exhaust_valves_per_cyl   1144 non-null   int64  
 14  fuel_injection          

In [9]:
df.nunique()

model                      750
model_index                456
displacement                41
cylinders                    8
gears                        8
transmission                 3
mpg                         38
aspiration                   2
lockup_torque_converter      2
drive                        4
max_ethanol                  3
recommended_fuel             3
intake_valves_per_cyl        2
exhaust_valves_per_cyl       2
fuel_injection               2
dtype: int64

# Keraksiz ustunnni tashab yuboramiz masalan -> Model Index

In [49]:
df.drop(columns=['model_index'],inplace=True)

In [73]:
class Preprocessing:
    def __init__(self,df):
        self.df=df
        
# Encoding qiluvchi
    def encodlovchi(df):
        for col in df.columns:
            if df[col].dtype == 'object':
                if df[col].nunique() <= 2:
                    new_df = pd.get_dummies(df[col], prefix='New', dtype='int')
                    df.drop(columns=[col],inplace=True)
                    df = pd.concat([df,new_df],axis=1)
                else:
                    df[col] = labeller.fit_transform(df[col])
        return df
    
# Scale qiluvchi
    def scaling(df):
        for cols in df.columns:
            if df[cols].dtype != 'object' and df[cols].name != 'mpg':
                df[cols] = minmax.fit_transform(df[[cols]])
        return df
    
# To'ldruvchi
    def Nan_Toldiruvchi(df):
        for cols in df.columns:
            if df[cols].isnull().any():
                if df[cols].dtype == 'object':
                    df[cols].fillna(df[cols].mode()[0], inplace=True)
                else:
                    df[cols].fillna(df[cols].mean(),inplace=True)
        return df           

In [55]:
df.head(3)

,model,displacement,cylinders,gears,transmission,mpg,drive,max_ethanol,recommended_fuel,intake_valves_per_cyl,exhaust_valves_per_cyl,New_Naturally Aspirated,New_Turbocharged/Supercharged,New_N,New_Y,New_Direct ignition,New_Multipoint/sequential ignition
0,7,3.5,6,9,2,21,3,10,1,2,2,0,1,0,1,1,0
1,0,1.8,4,6,2,28,1,10,1,2,2,0,1,0,1,1,0
2,34,5.2,10,7,2,17,3,15,0,2,2,1,0,0,1,1,0


# target qiymatga(mpg) juda kam korelatsya bo'gan ustunni tashlab yuboramiz -> Fuel Injection qiymatini

In [61]:
corr_df = df.corr()['mpg']
print(corr_df)

model                                 0.119912
displacement                         -0.740938
cylinders                            -0.713924
gears                                -0.385157
transmission                          0.244639
mpg                                   1.000000
drive                                -0.373795
max_ethanol                          -0.124257
recommended_fuel                      0.152029
intake_valves_per_cyl                 0.277959
exhaust_valves_per_cyl                0.291316
New_Naturally Aspirated               0.007729
New_Turbocharged/Supercharged        -0.007729
New_N                                 0.247939
New_Y                                -0.247939
New_Direct ignition                  -0.019907
New_Multipoint/sequential ignition    0.019907
Name: mpg, dtype: float64


In [63]:
df.drop(columns=['New_Direct ignition','New_Multipoint/sequential ignition'],inplace=True)

In [65]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1144 entries, 0 to 1143
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   model                          1144 non-null   int64  
 1   displacement                   1144 non-null   float64
 2   cylinders                      1144 non-null   int64  
 3   gears                          1144 non-null   int64  
 4   transmission                   1144 non-null   int64  
 5   mpg                            1144 non-null   int64  
 6   drive                          1144 non-null   int64  
 7   max_ethanol                    1144 non-null   int64  
 8   recommended_fuel               1144 non-null   int64  
 9   intake_valves_per_cyl          1144 non-null   int64  
 10  exhaust_valves_per_cyl         1144 non-null   int64  
 11  New_Naturally Aspirated        1144 non-null   int64  
 12  New_Turbocharged/Supercharged  1144 non-null   i